In [4]:
import pandas as pd
import numpy as np
from wordfreq import zipf_frequency

In [5]:
rt_df = pd.read_csv("data/SPRT_LogLin_216.csv")

gpt2 = pd.read_csv("data/surprisal_output/gpt2_wordlevel_batched.csv")
g270 = pd.read_csv("data/surprisal_output/google_gemma-3-270m_wordlevel_batched.csv")
g12b = pd.read_csv("data/surprisal_output/google_gemma-3-12b-pt_wordlevel_batched.csv")

gpt2_sub = gpt2[['ITEM', 'condition', 'cloze_surprisal', 'uni_surprisal_word', 'bi_surprisal_word']].rename(
    columns={'uni_surprisal_word': 'gpt2_uni_surprisal', 
             'bi_surprisal_word': 'gpt2_bi_surprisal'}
)
g270_sub = g270[['ITEM', 'condition', 'uni_surprisal_word', 'bi_surprisal_word']].rename(
    columns={'uni_surprisal_word': 'gemma270m_uni_surprisal', 
             'bi_surprisal_word': 'gemma270m_bi_surprisal'}
)
g12b_sub = g12b[['ITEM', 'condition', 'uni_surprisal_word', 'bi_surprisal_word']].rename(
    columns={'uni_surprisal_word': 'gemma12b_uni_surprisal', 
             'bi_surprisal_word': 'gemma12b_bi_surprisal'}
)

merged = pd.merge(rt_df, gpt2_sub, on=['ITEM', 'condition'], how='left')
merged = pd.merge(merged, g270_sub, on=['ITEM', 'condition'], how='left')
merged = pd.merge(merged, g12b_sub, on=['ITEM', 'condition'], how='left')

# Control variables
# A. Word Length (Characters)
merged['word_length'] = merged['critical_word'].str.len()

# B. Word Position 
merged['word_position'] = merged['position']

# C. Word Frequency (Zipf frequency: log10 scale, 1 to 8)
merged['word_frequency'] = merged['critical_word'].apply(lambda w: zipf_frequency(str(w), 'en'))

# Log Reading Time
merged['log_rt'] = np.log(merged['SUM_3RT_trimmed'])

# Squared Surprisal (Polynomial terms)
merged['cloze_surp_sq'] = merged['cloze_surprisal']**2

for model in ['gpt2', 'gemma270m', 'gemma12b']:
    merged[f'{model}_uni_sq'] = merged[f'{model}_uni_surprisal']**2
    merged[f'{model}_bi_sq'] = merged[f'{model}_bi_surprisal']**2

merged = merged.dropna(subset=['gpt2_uni_surprisal', 'SUM_3RT_trimmed', 'cloze_surprisal'])

merged.to_csv("data/master_modeling_data.csv", index=False)

print(f"Merge Complete! Total rows: {len(merged)}")

Merge Complete! Total rows: 46092
